# 문서 파싱 실습 — 26.09.29

이미 받은 파일로 실습합니다. **다운로드 코드가 없습니다.** 아래 `INPUT_DIR`에 문서 폴더를 지정한 뒤 위에서부터 실행하세요. 프로젝트 `.venv` 커널을 사용합니다.

1. 파일 분석
2. 파싱 불가 파일 제외
3. PDF 파싱
4. HWPX·DOCX 파싱
5. 문서별 문단 수·표 수·각 문단 첫 50글자 출력
6. 표 마크다운 변환 및 저장


In [59]:
from pathlib import Path
from collections import Counter
import sys
import hashlib
import unicodedata

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents)
             if (p / "backend/app/rag/local_parsers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("agent_practice 프로젝트 내부에서 실행하세요.")
if str(ROOT / "backend") not in sys.path:
    sys.path.insert(0, str(ROOT / "backend"))
from app.rag.local_parsers import HANDLERS, parse_local

DAY02 = ROOT / "sandbox/w5/day02"
# macOS의 한글 파일명 정규화 차이를 고려해 실제 폴더를 찾습니다.
INPUT_DIR = next((p for p in DAY02.iterdir()
                  if p.is_dir() and unicodedata.normalize("NFC", p.name) == "26.09.29_파싱실습"),
                 DAY02 / "26.09.29_파싱실습")
OUTPUT_DIR = DAY02 / "03_문서파싱실습_결과"
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"입력 폴더를 확인하세요: {INPUT_DIR}")
print("입력:", INPUT_DIR)
print("결과:", OUTPUT_DIR)

입력: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/26.09.29_파싱실습
결과: /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice/sandbox/w5/day02/03_문서파싱실습_결과


### 파일 형식 분석

- 입력 폴더의 모든 파일을 탐색
- 확장자별 파일 개수를 집계
- `.docx`, `.pdf`, `.hwpx` 파일 수를 각각 확인
- 그 외 확장자는 `그 밖`으로 합산

In [60]:
# 1. 파일 분석
from collections import Counter

SAMPLES = INPUT_DIR
ORDER = [".docx", ".pdf", ".hwpx"]

counts = Counter(
    p.suffix.lower()
    for p in SAMPLES.rglob("*")
    if p.is_file()
)
others = sum(n for ext, n in counts.items() if ext not in ORDER)

print(f"파일 {sum(counts.values())}개")
print(
    " " + " , ".join(f"{ext} {counts[ext]}" for ext in ORDER)
    + f" , 그 밖 {others}"
)

파일 8개
 .docx 0 , .pdf 2 , .hwpx 4 , 그 밖 2


### 2. 파싱 불가한 파일 제외
기존 `local_parsers.py`의 `HANDLERS`를 확인해 지원하지 않는 확장자를 제외합니다. 기존 파서가 지원하는 PDF·HWPX·DOCX를 모두 처리합니다.

이 단계는 확장자와 빈 파일만 검사합니다. 파일 손상이나 추출할 텍스트가 없는 문제는 3·4단계에서 기존 `parse_local()`의 반환값과 예외로 판별합니다. 원본은 삭제하지 않습니다.

In [61]:
candidates = {ext: [] for ext in HANDLERS}
excluded = []
parsed_docs = {}  # 기존 파서가 반환한 ParsedDoc 객체를 그대로 보관

for path in files:
    ext = path.suffix.lower()
    reason = None
    if ext not in HANDLERS:
        reason = "기존 local_parsers.py에서 지원하지 않는 형식"
    elif path.stat().st_size == 0:
        reason = "빈 파일"
    if reason:
        excluded.append({"file": str(path.relative_to(INPUT_DIR)), "reason": reason})
        print("제외:", path.name, "→", reason)
        continue
    candidates[ext].append(path)

for ext, paths in candidates.items():
    print(f"{ext} 후보: {len(paths)}개")

제외: 서울시 초미세먼지 경보 현황.json → 기존 local_parsers.py에서 지원하지 않는 형식
제외: 형사사법절차 전자화 촉진법(법률)(제22008호)(20261002).hwp → 기존 local_parsers.py에서 지원하지 않는 형식
.docx 후보: 0개
.pdf 후보: 2개
.hwpx 후보: 4개


### 3. PDF 파싱

- 입력 폴더에서 `.pdf` 파일을 순차적으로 탐색
- `PdfReader`를 사용해 PDF 파일을 읽음
- 각 페이지의 텍스트를 추출해 리스트 형태로 반환
- 탐색된 PDF 파일 개수와 파일명을 출력

In [62]:
# 3. pdf 파싱
def parse_pdf(path: Path) -> list[str]:
    reader = PdfReader(str(path))
    return [(page.extract_text() or "").strip() for page in reader.pages]

pdf_files = sorted(SAMPLES.glob('*.pdf'))

print(f"PDF 파일 개수 : {len(pdf_files)}개")
for pdf_file in pdf_files:
    print(f"가져온 파일 : {pdf_file.name}")

PDF 파일 개수 : 2개
가져온 파일 : 경영지도사 및 기술지도사에 관한 법률(법률)(제22031호)(20260929).pdf
가져온 파일 : 독도 등 도서지역의 생태계 보전에 관한 특별법(법률)(제21065호)(20260926).pdf


### 4. HWPX 파싱

- 입력 폴더에서 `.hwpx` 파일을 순차적으로 탐색
- HWPX 파일을 ZIP 형식으로 열어 내부 파일 목록을 확인
- 탐색된 HWPX 파일 개수와 파일명을 출력

In [63]:
# 4. hwpx 파싱
import zipfile

hwpx_files = sorted(SAMPLES.glob("*.hwpx"))
print(f"HWPX 파일 {len(hwpx_files)}개")

for HWPX in hwpx_files:
    with zipfile.ZipFile(HWPX) as zf:
        names = zf.namelist()

    print()
    print("가져온 파일: ", HWPX.name)

HWPX 파일 4개

가져온 파일:  가축전염병 예방법 시행규칙(농림축산식품부령)(제00793호)(20260929).hwpx

가져온 파일:  공항시설법 시행규칙(국토교통부령)(제01621호)(20260915).hwpx

가져온 파일:  시체 해부 및 보존 등에 관한 법률 시행령(대통령령)(제36725호)(20260929).hwpx

가져온 파일:  형사소송법(법률)(제22009호)(20261002).hwpx


### 파싱된 문서 문단 개수, 표의 개수, 문단 내용 첫 50글자

### 5-a PDF 표 개수 확인

- 각 PDF 파일의 텍스트 파싱 여부를 확인
- `pdfplumber`를 사용해 페이지별 표 개수를 탐색
- 표가 존재하는 페이지와 해당 표 개수를 출력
- 파일별 전체 표 개수를 합산해 출력

In [64]:
# PDF 표 개수 세기
from pypdf import PdfReader
import pdfplumber

for path in pdf_files : 
    pdf = parse_pdf(path)
    print(f"{path.name} 파싱 완료")
   
    found = 0
    with pdfplumber.open(str(path)) as pdf:
        print("--표는 아직파싱 안함--")
        for n, page in enumerate(pdf.pages, 1):
            k = len(page.extract_tables())
            if k:
                print(f'p.{n:2d} : {k}개')
                found += k
    print(f'PDF 표 합계 : {found}')
    print()

경영지도사 및 기술지도사에 관한 법률(법률)(제22031호)(20260929).pdf 파싱 완료
--표는 아직파싱 안함--
PDF 표 합계 : 0

독도 등 도서지역의 생태계 보전에 관한 특별법(법률)(제21065호)(20260926).pdf 파싱 완료
--표는 아직파싱 안함--
PDF 표 합계 : 0



### 5-b HWPX 표 개수 확인

- 입력 폴더의 `.hwpx` 파일을 순차적으로 탐색
- `parse_hwpx()`를 사용해 각 파일을 파싱
- 파싱 결과의 `table_count`를 이용해 표 개수를 확인
- 파일별 전체 표 개수를 출력

In [65]:
import re
import zipfile
from xml.etree import ElementTree as ET

ns_p = "{http://www.hancom.co.kr/hwpml/2011/paragraph}"

hwpx_files = sorted(SAMPLES.glob("*.hwpx"))
for file in hwpx_files:
    table_count = 0
    with zipfile.ZipFile(file) as zf:
        sections = sorted(n for n in zf.namelist() if re.match(r"Contents/section\d+\.xml$", n))
        for name in sections:
            root = ET.fromstring(zf.read(name))
            table_count += len(root.findall(f".//{ns_p}tbl"))

    print(f"{file.name} 파싱 완료 (섹션 {len(sections)}개)")
    print(f'HWPX 표 합계 : {table_count}')

가축전염병 예방법 시행규칙(농림축산식품부령)(제00793호)(20260929).hwpx 파싱 완료 (섹션 1개)
HWPX 표 합계 : 2
공항시설법 시행규칙(국토교통부령)(제01621호)(20260915).hwpx 파싱 완료 (섹션 1개)
HWPX 표 합계 : 2
시체 해부 및 보존 등에 관한 법률 시행령(대통령령)(제36725호)(20260929).hwpx 파싱 완료 (섹션 1개)
HWPX 표 합계 : 2
형사소송법(법률)(제22009호)(20261002).hwpx 파싱 완료 (섹션 1개)
HWPX 표 합계 : 2


### 5-c PDF 문단 확인

- 각 PDF 파일에서 텍스트가 존재하는 페이지만 추출
- 추출된 문단 개수를 파일별로 출력
- 각 문단의 앞부분을 최대 50자까지 미리 확인
- 문단 내 줄바꿈과 불필요한 공백을 정리하여 출력

In [66]:
for PDF in pdf_files:
    para = [text for text in parse_pdf(PDF) if text]

    print(f"{PDF.name} : 문단 {len(para)}개")
    for text in para:
        body = text.split("\n", 2)[-1]
        print("  -", " ".join(body.split())[:50])
    print("-"*50)

경영지도사 및 기술지도사에 관한 법률(법률)(제22031호)(20260929).pdf : 문단 8개
  - 경영지도사 및 기술지도사에 관한 법률 ( 약칭: 경영기술지도사법 ) [시행 2026. 9.
  - ⑤ 제1항에 따른 지도사자격시험의 응시 자격, 시험 과목, 시험 방법 및 제3항에 따른 시
  - 제10조(등록취소와 업무정지) ① 중소벤처기업부장관은 제8조에 따라 등록을 한 지도사가 다
  - 제19조(비밀엄수) 지도사나 그 사무직원 또는 지도사였거나 그 사무직원이었던 사람은 다른 
  - 3. 주사무소와 분사무소의 소재지 4. 사원과 이사의 성명, 주민등록번호 및 주소 5. 출
  - 제29조(다른 법인에 대한 출자의 제한 등) ① 지도법인은 자기자본에 대통령령으로 정하는 
  - 3. 업무정지명령을 위반하여 업무를 수행한 경우 4. 제27조제3항에 따른 중소벤처기업부장
  - 2. 제19조(제37조제1항에서 준용하는 경우를 포함한다)를 위반하여 직무상 알게 된 비밀
--------------------------------------------------
독도 등 도서지역의 생태계 보전에 관한 특별법(법률)(제21065호)(20260926).pdf : 문단 6개
  - 독도 등 도서지역의 생태계 보전에 관한 특별법 ( 약칭: 도서생태계법 ) [시행 2026.
  - 제5조(특정도서보전 기본계획) ① 기후에너지환경부장관은 특정도서의 자연생태계등을 보전하기 
  - ④ 토지의 소유자등은 정당한 사유 없이 제1항에 따른 조사행위를 거부, 방해 또는 기피할 
  - ② 기후에너지환경부장관은 제1항에 따른 허가를 할 때 필요한 조건이나 기한을 붙일 수 있다
  - [전문개정 2011. 7. 28.] 제14조(벌칙) 제8조제1항제1호부터 제11호까지 및 
  - 제8조 생략
-----------------------------------------

### 5-d hwpx 문단 확인

In [67]:
for file in hwpx_files:
    result = parse_hwpx(file)

    para = [
        block for block in result.blocks
        if block.kind == "조항"
    ]

    print(f"{file.name} : 문단 {len(para)}개")

    text = " ".join(block.text.strip() for block in para)

    print("  -", " ".join(text.split())[:50])

    print("-" * 50)

가축전염병 예방법 시행규칙(농림축산식품부령)(제00793호)(20260929).hwpx : 문단 871개
  - 가축전염병 예방법 시행규칙 농림축산식품부(방역정책과-가축의 방역) 044-201-2519,
--------------------------------------------------
공항시설법 시행규칙(국토교통부령)(제01621호)(20260915).hwpx : 문단 525개
  - 공항시설법 시행규칙 국토교통부(공항정책과) 044-201-4337, 4265 제1조(목적)
--------------------------------------------------
시체 해부 및 보존 등에 관한 법률 시행령(대통령령)(제36725호)(20260929).hwpx : 문단 83개
  - 시체 해부 및 보존 등에 관한 법률 시행령 보건복지부(생명윤리정책과) 044-202-261
--------------------------------------------------
형사소송법(법률)(제22009호)(20261002).hwpx : 문단 1892개
  - ※ 본 법령은 공포 후 시행전 개정된 사항을 모두 포함한 법령이며, 개별 시행일별로 공포된
--------------------------------------------------


### 6 표를 마크다운으로 변환

In [72]:
from pathlib import Path
import pdfplumber

from app.rag.local_parsers import parse_hwpx


# 저장 폴더
MARKDOWN_DIR = ROOT / "sandbox/w5/day02/markdownResult"
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)

In [73]:
for path in hwpx_files:
    result = parse_hwpx(path)

    tables = [
        block.text
        for block in result.blocks
        if block.kind == "표"
    ]

    parts = [
        f"## 표 {i}\n\n{table}"
        for i, table in enumerate(tables, 1)
    ]

    out_path = MARKDOWN_DIR / f"{path.stem}.md"

    text = "\n\n".join(parts)

    out_path.write_text(text, encoding="utf-8")

    print(f"{path.name} → {out_path.name} / 표 {len(tables)}개")

가축전염병 예방법 시행규칙(농림축산식품부령)(제00793호)(20260929).hwpx → 가축전염병 예방법 시행규칙(농림축산식품부령)(제00793호)(20260929).md / 표 1개
공항시설법 시행규칙(국토교통부령)(제01621호)(20260915).hwpx → 공항시설법 시행규칙(국토교통부령)(제01621호)(20260915).md / 표 1개
시체 해부 및 보존 등에 관한 법률 시행령(대통령령)(제36725호)(20260929).hwpx → 시체 해부 및 보존 등에 관한 법률 시행령(대통령령)(제36725호)(20260929).md / 표 1개
형사소송법(법률)(제22009호)(20261002).hwpx → 형사소송법(법률)(제22009호)(20261002).md / 표 1개
